# Đồ thị 08 Heatmap tương quan sáu chỉ số định lượng

**Võ Minh Huy — 2311110117**

Tính Pearson giữa sáu cột trên đủ 299 hồ sơ: age, creatinine_phosphokinase, ejection_fraction, platelets, serum_creatinine và serum_sodium. Dùng platelets đã làm sạch, không dùng hai cột làm mịn.

Ma trận này có kích thước 6×6, mỗi ô là tương quan giữa hai thuộc tính trên 299 hồ sơ. Khác với ma trận 3×3 giữa ba hồ sơ ở mục 3f. Không đưa MaHoSo, MaBoolean, nhãn DEATH_EVENT hoặc mã nhóm tuổi vào ma trận này.


In [ ]:
from pathlib import Path
import sys
import subprocess
import numpy as np
import pandas as pd
try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "matplotlib"])
    import matplotlib.pyplot as plt

thu_muc = Path.cwd()
if thu_muc.name == "python":
    thu_muc = thu_muc.parent
du_lieu = pd.read_csv(thu_muc / "VoMinhHuy_Heart Failure Prediction_after_Python.csv")
cot = ["age", "creatinine_phosphokinase", "ejection_fraction",
    "platelets", "serum_creatinine", "serum_sodium"]
chi_so = du_lieu[cot].apply(pd.to_numeric,errors="raise")
if len(chi_so) != 299 or chi_so.isna().any().any() or not np.isfinite(chi_so).all().all():
    raise ValueError("Cần đủ 299 hồ sơ với sáu chỉ số hữu hạn, không thiếu.")
if (chi_so.std(ddof=1) <= 0).any():
    raise ValueError("Không tính Pearson cho cột hằng.")
thu_muc_kq = thu_muc / "ket_qua/21_do_thi/08_heatmap_tuong_quan"
thu_muc_kq.mkdir(parents=True, exist_ok=True)
quy_mo = pd.DataFrame({"SoHoSo":[len(chi_so)],"SoThuocTinh":[len(cot)],
    "SoCapKhacNhau":[len(cot)*(len(cot)-1)//2],"SoGiaTriThieu":[int(chi_so.isna().sum().sum())]})
quy_mo


In [ ]:
tuong_quan = chi_so.corr(method="pearson")
doi_chieu_numpy = np.corrcoef(chi_so.to_numpy(),rowvar=False)
assert np.allclose(tuong_quan.to_numpy(),doi_chieu_numpy,atol=1e-12,rtol=0)
assert np.allclose(tuong_quan,tuong_quan.T,atol=1e-12,rtol=0)
assert np.allclose(np.diag(tuong_quan),1)
assert np.isfinite(tuong_quan.to_numpy()).all()
assert np.abs(tuong_quan.to_numpy()).max() <= 1 + 1e-12
tuong_quan.round(3)


In [ ]:
# Chỉ lấy tam giác trên, không tính đường chéo hoặc đếm lặp hai chiều.
cac_cap = []
for i in range(len(cot)):
    for j in range(i+1,len(cot)):
        r = float(tuong_quan.iloc[i,j])
        cac_cap.append({"ThuocTinh1":cot[i],"ThuocTinh2":cot[j],"PearsonR":r,"TriTuyetDoiR":abs(r)})
bang_cap = pd.DataFrame(cac_cap).sort_values("TriTuyetDoiR",ascending=False,ignore_index=True)
assert len(bang_cap) == 15
bang_cap.head(5).round(3)


In [ ]:
plt.rcParams.update({"font.family":"DejaVu Sans", "font.size":11})
nhan = ["Tuổi", "CPK", "EF", "Tiểu cầu", "Creatinine", "Natri"]
fig, ax = plt.subplots(figsize=(9.5,8))
fig.subplots_adjust(top=0.80,bottom=0.15,left=0.17,right=0.88)
fig.text(0.12,0.94,"Tương quan giữa sáu chỉ số định lượng",fontsize=17,weight="bold")
fig.text(0.12,0.90,"299 hồ sơ sau tiền xử lý · Pearson r · Thang màu cố định từ −1 đến 1",color="#555555",fontsize=10)
anh = ax.imshow(tuong_quan.to_numpy(),cmap="RdBu_r",vmin=-1,vmax=1,interpolation="nearest")
ax.set_xticks(np.arange(6),labels=nhan,rotation=35,ha="right")
ax.set_yticks(np.arange(6),labels=nhan)
for i in range(6):
    for j in range(6):
        r = tuong_quan.iloc[i,j]
        ax.text(j,i,f"{r:.2f}",ha="center",va="center",
            color="white" if abs(r)>0.6 else "#222222",fontsize=11)
ax.set_xticks(np.arange(-0.5,6,1),minor=True)
ax.set_yticks(np.arange(-0.5,6,1),minor=True)
ax.grid(which="minor",color="white",linewidth=1.5)
ax.tick_params(which="minor",bottom=False,left=False)
for spine in ax.spines.values():
    spine.set_visible(False)
cbar=fig.colorbar(anh,ax=ax,fraction=0.045,pad=0.04,ticks=[-1,-0.5,0,0.5,1])
cbar.set_label("Pearson r")
fig.savefig(thu_muc_kq / "08_heatmap_tuong_quan.png",dpi=300,bbox_inches="tight",facecolor="white")
quy_mo.to_csv(thu_muc_kq / "quy_mo_python.csv",index=False,encoding="utf-8-sig")
tuong_quan.to_csv(thu_muc_kq / "tuong_quan_python.csv",index=True,index_label="ThuocTinh",encoding="utf-8-sig")
bang_cap.to_csv(thu_muc_kq / "cac_cap_python.csv",index=False,encoding="utf-8-sig")
plt.show()


## Nhận xét

Đường chéo bằng 1 vì mỗi biến tương quan với chính nó. Đọc các ô ngoài đường chéo và bảng các cặp theo |r|; dấu dương chỉ quan hệ tuyến tính cùng chiều, dấu âm chỉ quan hệ ngược chiều. Bảng xếp hạng không phải kiểm định ý nghĩa thống kê.

Pearson gần 0 chỉ cho thấy quan hệ tuyến tính yếu, không chứng minh hai biến độc lập. Tương quan không chứng minh nhân quả và không dùng riêng heatmap để kết luận tình trạng bệnh. Thang màu giữ cố định −1 đến 1 để không phóng đại các hệ số nhỏ.

Không cần chuẩn hóa đơn vị trước khi tính Pearson vì hệ số bất biến khi đổi thang đo tuyến tính dương. Kết quả dùng dữ liệu đã điền thiếu và xử lý nhiễu, nên có thể chịu ảnh hưởng của tiền xử lý. Không cần nhãn nên hồ sơ chờ xác minh DEATH_EVENT vẫn tham gia; file after giữ nguyên.
